<a href="https://colab.research.google.com/github/temesgenaddise/cosc-650-applied-llm-systems/blob/main/Ass_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Amharic–English Tokenization Analysis With Tiktoken

This notebook compares the supplied Amharic passage with it's English translation using GPT-4 tokenizer (cl100k_base) and GPT-4o tokenizer (o200k_base).

##Part 1: Labiraries

In [1]:
!pip install tiktoken
import os, pathlib
os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.') / '.tiktoken_cache').resolve())
os.makedirs(os.environ['TIKTOKEN_CACHE_DIR'], exist_ok=True)

import tiktoken
gpt4  = tiktoken.get_encoding('cl100k_base')
gpt4o = tiktoken.get_encoding('o200k_base')
print('tiktoken', tiktoken.__version__, '- encoders ready (cl100k_base, o200k_base)')

tiktoken 0.14.0 - encoders ready (cl100k_base, o200k_base)


In [2]:
def count_tokens(text, enc):
    return len(enc.encode(text))

def show_split(word, enc=gpt4):
    ids = enc.encode(word)
    pieces = [enc.decode([i]) for i in ids]
    print(f'{word!r:18s} -> {len(ids)} token(s): {pieces}')

##Part 2: Token and character analysis

In [3]:
english_text = '''Hope is not a wish or a longing, but a precious gift of God in which we can rely and trust. Hope is not limited to this world, not limited by matter, but by the unseen; it is the assurance that we are partakers of the divine glory of the kingdom of God in the world to come, which is not seen now, but is established in heaven and is not destroyed. Hope is the power of faith that a life built on faith will bear fruit. Hope is the power of faith that assures us that even if we fall into the deepest darkness and gravest sin in our lives, God's love, goodness, forgiveness, and light are in us and that we can do for us what we cannot do for ourselves. Hope, like faith, is unseen. Hope is not hope that is seen; for who hopes for what he sees? But if we hope for what we do not see, we wait patiently. Amen.'''
Amharic_text = '''ተስፋ ምኞትና ናፍቆት ሳይሆን አለኝታና መተማመን የሚገኝበት የአምላክ ውድ ስጦታ ነው። ተስፋ በዚህ ዓለም ብቻ የማይወሰን፣ በቁስ የማይገደብ የማይታየውን የምናይበት ፤ አሁን ባይታይም በሰማያዊ ስፍራ የተረጋገጠና የማይጠፋ በሚመጣው ዓለም የእግዚአብሔር መንግሥት ከመለኮታዊ ክብሩ የጸጋ ተካፋዮች ለመሆናችን ማረጋገጫን ነው።ተስፋ በእምነት ላይ የተገነባው ሕይወት ፍሬውን እንደሚያፈራ ማረጋገጫ የእምነት ኃይል ነው።ተስፋ በሕይወታችን ምንም እንኳን ድቅድቅ ጨለማ እና የከፋ ኃጢአት ውስጥ ብንወድቅም ፣ የእግዚአብሔር ፍቅር፣ ቸርነት፣ ይቅርታ፣ ብርሃን በእኛ እና እኛ ራሳችን ማድረግ የማንችለውን ለእኛ ለማድረግ የሚያስችል መተማመን እንዳለን የምናስረግጥበት ኃይል ነው።ተስፋ እንደ እምነት ሁሉ የማይታይ ነው። ተስፋ የሚደረግበቱ ነገር ቢታይ ተስፋ አይደለም፤ የሚያየውንማ ማን ተስፋ ያደርገዋል? የማናየውን ግን ተስፋ ብናደርገው በትዕግሥት እንጠባበቃለን። አሜን '''

print('English words:', len(english_text.split()))
print('Amharic words:', len(Amharic_text.split()))

def report(label, text):
    print(f'{label:9s} | chars {len(text):4d} | GPT-4 {count_tokens(text, gpt4):4d} | GPT-4o {count_tokens(text, gpt4o):4d}')

report('English', english_text)
report('Amharic', Amharic_text)

tax_gpt4  = count_tokens(Amharic_text, gpt4)  / count_tokens(english_text, gpt4)
tax_gpt4o = count_tokens(Amharic_text, gpt4o) / count_tokens(english_text, gpt4o)
print(f'\nMultilingual tax  GPT-4: {tax_gpt4:.2f}x   GPT-4o: {tax_gpt4o:.2f}x')

English words: 166
Amharic words: 100
English   | chars  809 | GPT-4  191 | GPT-4o  190
Amharic   | chars  542 | GPT-4 1316 | GPT-4o  983

Multilingual tax  GPT-4: 6.89x   GPT-4o: 5.17x


The multilingual tax is defined as non-English tokens / English tokens for both tokenozer. A value of 6.89 and 5.174 in above result means the same bilingual content representation consumes 6.89 and 5.174 times as many input tokens in Amharic than in English under that tokenizer.

##Part 3: Evaluate with real figures

In [4]:
CTX = 128_000
en = count_tokens(english_text, gpt4)
fo = count_tokens(Amharic_text, gpt4)
print(f'A {CTX:,}-token window holds about {CTX//en:,} English copies and {CTX//fo:,} Amharic copies of your passage.')
print(f'Per-request cost multiplier for the Amharic language: {fo/en:.2f}x (billing is per token).')

A 128,000-token window holds about 670 English copies and 97 Amharic copies of your passage.
Per-request cost multiplier for the Amharic language: 6.89x (billing is per token).


The copy calculation uses floor division because only complete copies count. The cost calculation assumes the provider bills both languages at the same input-token price. Under that assumption, the token ratio is exactly the per-request input-cost multiplier. It excludes output tokens, cached-token discounts, and fixed fees. The per request cost mutilier for Amharic is 589% and 417% than per request for English in GPT-4 and GPT=4o respectively.

##Three specific splits showing English-corpus bias

In [5]:
import pandas as pd

def byte_pieces(enc, text):

    return [repr(enc.decode_single_token_bytes(token_id)) for token_id in enc.encode(text)]

def show_split(word, enc=gpt4):
    ids = enc.encode(word)
    pieces = [enc.decode([i]) for i in ids]
    print(f'{word!r:18s} -> {len(ids)} token(s): {pieces}')

examples = [
    ("ተስፋ", "hope"),
    ("እግዚአብሔር", "God"),
    ("መተማመን", "trust"),
]

encodings = {
    'gpt4': gpt4,
    'gpt4o': gpt4o,
}

split_rows = []
for tokenizer, enc in encodings.items():
    for am_word, en_equiv in examples:
        split_rows.append({
            "Tokenizer": tokenizer,
            "Amharic / English": f"{am_word} / {en_equiv}",
            "Amharic token count": len(enc.encode(am_word)),
            "Amharic token byte pieces": " | ".join(byte_pieces(enc, am_word)),
            "English token count": len(enc.encode(en_equiv)),
            "English token byte pieces": " | ".join(byte_pieces(enc, en_equiv)),
        })

splits = pd.DataFrame(split_rows)
pd.set_option("display.max_colwidth", None)
print(splits.to_string(index=False))

Tokenizer Amharic / English  Amharic token count                                                                                                                                                                                       Amharic token byte pieces  English token count English token byte pieces
     gpt4        ተስፋ / hope                    9                                                                                                                         b'\xe1' | b'\x89' | b'\xb0' | b'\xe1' | b'\x88' | b'\xb5' | b'\xe1' | b'\x8d' | b'\x8b'                    1                   b'hope'
     gpt4     እግዚአብሔር / God                   21 b'\xe1' | b'\x8a' | b'\xa5' | b'\xe1' | b'\x8c' | b'\x8d' | b'\xe1' | b'\x8b' | b'\x9a' | b'\xe1' | b'\x8a' | b'\xa0' | b'\xe1' | b'\x89' | b'\xa5' | b'\xe1' | b'\x88' | b'\x94' | b'\xe1' | b'\x88' | b'\xad'                    1                    b'God'
     gpt4     መተማመን / trust                   13                                        

The b'...' values above are the actual token bytes; adjoining them reconstructs the displayed word. This is stronger evidence of fragmentation than decoding each token separately, which would replace incomplete characters. These examples reveal the corpus bias because common English equivalents are represented compactly, while the Amharic words require many more pieces. GPT-4o's newer vocabulary reduces the gap, but does not remove it.

##Part 4: Find One failure and Explain it

In [6]:
composed = "é"                 # U+00E9
decomposed = "e\u0301"       # U+0065 + U+0301; visually also “é”

failure_rows = []
for tokenizer, enc in encodings.items():
    for label, value in [("NFC composed", composed), ("NFD decomposed", decomposed)]:
        failure_rows.append({
            "Tokenizer": tokenizer,
            "Form": label,
            "Displayed text": value,
            "Code points": " ".join(f"U+{ord(ch):04X}" for ch in value),
            "Characters": len(value),
            "Tokens": len(enc.encode(value)),
            "Actual token byte pieces": " | ".join(byte_pieces(enc, value)),
        })

The two strings look identical on screen, but their Unicode representations differ. Byte-pair tokenizers operate on encoded byte patterns, not on visual glyphs. The decomposed form contains a separate combining acute accent, so it can require more tokens.

##Mitigation:

In [7]:
import unicodedata

normalized = unicodedata.normalize("NFC", decomposed)
print("Visually equal after NFC:", normalized == composed)
for tokenizer, enc in encodings.items():
    print(tokenizer, "before:", len(enc.encode(decomposed)), "after NFC:", len(enc.encode(normalized)))

Visually equal after NFC: True
gpt4 before: 2 after NFC: 1
gpt4o before: 2 after NFC: 1
